# 07 — Feedback & GEPA: from scalar loss to textual gradient

**What you'll learn**

- Why a scalar metric wastes most of what your evaluation actually *knows* — and GEPA's fix: metrics that return **text**.
- The 5-argument feedback-metric contract (enforced at construction!) and `dspy.Prediction(score=..., feedback=...)`.
- The GEPA loop: Pareto candidate pool → reflective mutation by a strong LM → merge.
- The paper ↔ code map for *GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning* (Agrawal et al., 2025).
- The course ritual: one honest number on the sealed test split.

*Time: ~30 min. Cost: ~$0.40–0.80 (GEPA uses a strong reflection model). Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, browsable any time with `obs.mlflow_ui()`. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()

Boilerplate — same code as notebooks 03 and 04. Skim and run.

In [ ]:
# === boilerplate: aldervane micro-world — same code as notebook 03 ===
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

In [ ]:
# === boilerplate: data + metric — same code as notebook 04 ===
QA = json.loads(Path("data/qa.json").read_text())

def to_examples(rows):
    return [dspy.Example(**row).with_inputs("question") for row in rows]

train, dev, test = (to_examples(QA[s]) for s in ("train", "dev", "test"))

def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

In [ ]:
evaluate = dspy.Evaluate(devset=dev, metric=em_metric, num_threads=8, display_progress=True)

baseline_score = evaluate(HopQA()).score
baseline_score

## The information we've been throwing away

Think about what your notebook-04 error analysis *knew* about each failure: which gold passage was never retrieved, that the hop-2 query restated the question, that the answer named the wrong Venn, that a correct answer failed by being a full sentence. Rich, causal, actionable knowledge.

Now think about what the optimizers of notebooks 05–06 *received* from all of it: `0.0`.

A scalar can steer a search, but it cannot say *why* — every diagnosis is compressed into one bit and the search rediscovers the causes by brute force. GEPA's move: let the metric hand back the diagnosis **as text**, and put an LLM in the loop whose job is to *read* failures and rewrite the failing module's instructions accordingly. If demos were pseudo-labels and MIPROv2 was Bayesian search, this is the closest prompts get to a gradient — a direction, expressed in language.

## The feedback metric: a new contract

GEPA metrics take **five** positional arguments — `(gold, pred, trace, pred_name, pred_trace)`. The last two arrive when GEPA is reflecting on one specific module: its name, and just its slice of the trace, so your feedback can address the actual culprit. Return either a plain float, or `dspy.Prediction(score=..., feedback=...)`.

Everything we need is already lying around: `gold.gold_ids` (which passages contain the evidence) and our program's self-reported `queries` and `retrieved_ids`. The notebook-04 diagnosis, mechanized:

In [ ]:
def em_feedback_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """Exact-match score, plus a textual diagnosis GEPA's reflection LM can read."""
    score = float(norm(pred.answer) == norm(gold.answer))
    notes = []

    missing = sorted(set(gold.gold_ids) - set(pred.retrieved_ids))
    if missing:
        titles = [p["title"] for p in CORPUS if p["id"] in missing]
        notes.append(
            f"Retrieval failure: gold passage(s) {missing} ({', '.join(titles)}) were never "
            f"retrieved. Queries used: {pred.queries}. The second query must name the bridge "
            "entity discovered in hop 1 (a person, place, or institution), not restate the question."
        )
    elif not score:
        notes.append(
            f"All evidence was retrieved, yet {pred.answer!r} != {gold.answer!r}. Answer with "
            "the shortest exact phrase from the passages: no full sentences, no articles, "
            "no added words. Beware lookalike entities (two Venns, two Rasks, two foundries)."
        )
    else:
        notes.append("Correct.")

    if pred_name is not None:
        notes.append(f"(Feedback target: module {pred_name!r}.)")

    return dspy.Prediction(score=score, feedback=" ".join(notes))

The contract is not a suggestion — GEPA checks your metric's call signature **when you construct the optimizer**, so a notebook-04-style 2-or-3-argument metric fails years before it could fail silently:

In [ ]:
try:
    dspy.GEPA(metric=em_metric, auto="light", reflection_lm=dspy.LM(STRONG_MODEL))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

> **What you should see (deterministic):** an immediate error naming the expected metric signature — at *construction*, before any budget is spent. Contrast this with the usual LLM-pipeline failure mode (everything "works", quality is mysteriously bad). Typed contracts, enforced early, are a running DSPy theme.

## Run it

Two knobs deserve a word before the cell:

- **`reflection_lm` is required, and should be strong.** One model *runs* the task (cheap, many calls); another *reads failures and writes prompts* (strong, few calls). Spend where judgment lives.
- **Budget: exactly one of** `auto=`, `max_full_evals=`, `max_metric_calls=`. Evolutionary search would happily spend forever; you must cap it, and GEPA refuses to guess which cap you meant.

In [ ]:
reflection_lm = dspy.LM(STRONG_MODEL, temperature=1.0, max_tokens=8000)

gepa = dspy.GEPA(
    metric=em_feedback_metric,
    auto="light",                              # budget preset (or max_full_evals= / max_metric_calls=)
    reflection_lm=reflection_lm,
    reflection_minibatch_size=3,
    candidate_selection_strategy="pareto",
    component_selector="round_robin",
    use_merge=True,
    track_stats=True,                          # keep the full evolution record
)

gepa_prog = gepa.compile(HopQA(), trainset=train, valset=dev)

While it runs, here is the loop your tokens are funding:

```
   candidate pool of programs          (starts with just your HopQA)
        |
        v
   select a parent from the PARETO FRONT        <- candidate_selection_strategy="pareto":
        |                                          a candidate survives by being best on
        |                                          >= 1 validation question — diverse
        |                                          strategies are protected, not averaged away
        v
   pick ONE module to evolve                     <- component_selector="round_robin"
        |
        v
   run a small minibatch of train questions      <- reflection_minibatch_size=3
   collect traces + your metric's FEEDBACK text
        |
        v
   reflection LM reads trace + feedback and      <- reflection_lm
   REWRITES that module's instructions
        |
        v
   child program joins the pool; occasionally,
   MERGE two good lineages into one child        <- use_merge=True
        |
        +----> repeat until the budget is spent  <- auto / max_full_evals / max_metric_calls
```

The Pareto detail is the paper's quiet masterstroke, and worth internalizing: selecting only the best-average candidate collapses the pool onto one strategy and stalls; keeping everything that wins *somewhere* preserves the building blocks that merges and mutations later combine.

In [ ]:
gepa_score = evaluate(gepa_prog).score
print(f"baseline {baseline_score} -> GEPA {gepa_score}")

> **What you should see:** a score competitive with — often above — your best from notebooks 05–06, and reflection logs along the way showing candidates being proposed and scored.
>
> | program | dev EM (16 q) | expect | notebook |
> |---|---|---|---|
> | `HopQA()` zero-shot | *yours* | ~25–55 | 04 |
> | + `BootstrapFewShot` | *yours* | +10–25 | 05 |
> | + `BootstrapRS` | *yours* | ≥ bootstrap | 05 |
> | + `COPRO` | *yours* | ±10 | 06 |
> | + `MIPROv2 (light)` | *yours* | ≥ bootstrap-RS | 06 |
> | + `GEPA (light)` | *yours* | ≥ MIPROv2, ties common | 07 |
>
> Same toy-scale honesty as before: ties happen at n=16. The differentiating evidence in the paper is at scale — GEPA reaching or beating RL-style optimization with a fraction of the rollouts, *because each rollout teaches more than one bit*.

In [ ]:
for name, predictor in gepa_prog.named_predictors():
    print(f"--- {name} ---")
    print(predictor.signature.instructions, "\n")

> **What you should see — and actually read:** evolved instructions that are typically *long, specific, rubric-like paragraphs*: naming the bridge-entity strategy for `hop2`, the shortest-exact-phrase rule and the lookalike-entity traps for `answer`. Compare them with the feedback strings your metric emits — you will recognize your own diagnoses, generalized into policy. That traceability (metric feedback → reflection → instruction text) is the paper's headline behavior, reproduced on your machine.

In [ ]:
r = gepa_prog.detailed_results          # exists because track_stats=True
print("candidates tried:  ", len(r.candidates))
print("val scores:        ", [round(s, 3) for s in r.val_aggregate_scores])
print("best index:        ", r.best_idx)
print("total metric calls:", r.total_metric_calls)

> **What you should see:** the evolution record — every candidate program, its aggregate validation score, and the budget actually spent. With `track_stats=True` GEPA also records per-question bests; the docstring's party trick is using GEPA as a pure *inference-time* search (`valset=your_batch, track_best_outputs=True` → `detailed_results.best_outputs_valset` holds the best output per question). File under "the optimizer is also a search engine".

## Paper ↔ code: GEPA

*GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning* (Agrawal et al., 2025, [arXiv:2507.19457](https://arxiv.org/abs/2507.19457)):

| Paper concept | Plain English | Where it lives in code |
|---|---|---|
| Compound AI system Φ with module parameters | Your pipeline; each module's prompt is an evolvable gene | `dspy.Module` + `named_predictors()` |
| Feedback function μ_f | Evaluator returns a score **and** a diagnostic text | the 5-arg metric returning `dspy.Prediction(score=..., feedback=...)` (2-arg metrics rejected at construction) |
| Reflective prompt mutation | A strong LM reads rollout traces + feedback and rewrites one module's instructions | `reflection_lm=`, `reflection_minibatch_size=` |
| Pareto-based candidate selection | Keep any candidate that is best on ≥1 task instance; protect diverse strategies | `candidate_selection_strategy="pareto"` (vs `"current_best"`) |
| Module selection for mutation | Which module gets evolved this round | `component_selector="round_robin"` (or `"all"`) |
| System-aware merge (crossover) | Combine complementary lineages into one child | `use_merge=True`, `max_merge_invocations=` |
| Rollout / metric-call budget B | Total evaluations the search may spend | exactly one of `auto=` / `max_full_evals=` / `max_metric_calls=` |
| Candidate pool & evolution record | The genetic tree, per-candidate scores, per-instance bests | `track_stats=True` → `optimized.detailed_results` |

## The ritual: one honest number

The toy thread ends here, so we finally open the vault. The test split (8 questions) has never been touched — not by an optimizer, not by our eyeballs. One evaluation, of one program (your leaderboard's best), and we look at the result *once*:

In [ ]:
test_evaluate = dspy.Evaluate(devset=test, metric=em_metric, num_threads=8, display_progress=True)

best_program = gepa_prog        # <- set this to whichever program leads YOUR leaderboard
print("sealed test EM:", test_evaluate(best_program).score)

> **What you should see:** a number in the neighborhood of your dev score — but with n=8, each question is 12.5 points, so "neighborhood" is doing heavy lifting. What the ritual actually teaches: dev scores drift upward under optimization pressure (you tuned *on* dev — five optimizers' worth of selection), and only data that stayed sealed can price that drift. At real scale (capstone), the same ritual with real statistical teeth.

## Recap

| Concept | One-liner |
|---|---|
| feedback metric | `(gold, pred, trace, pred_name, pred_trace)` → `Prediction(score=..., feedback=...)` |
| enforcement | wrong metric arity fails at `dspy.GEPA(...)` construction, not mid-run |
| reflection | strong `reflection_lm` reads traces + feedback, rewrites one module's instructions |
| Pareto pool | best-on-≥1-example survives; diversity is the search's raw material |
| merge | crossover of complementary lineages (`use_merge=True`) |
| budget | exactly one of `auto` / `max_full_evals` / `max_metric_calls` |
| sealed test | opened once; the only unbiased number you own |

## Exercises

1. The ablation that matters: rerun GEPA with a metric that returns *only the float* (keep the 5-arg signature, drop the feedback text) at a small fixed budget (`max_metric_calls=150`). Compare against the feedback version at the same budget. You are measuring what the *text* is worth.
2. Swap `component_selector="round_robin"` for `"all"` (every module evolved each round). Same budget — better or worse here, and what does that say about credit assignment on a 3-module program?
3. Extend `em_feedback_metric` to critique *near-miss verbosity* separately (correct entity, extra words) from *wrong entity*, using `pred_name` to route the style advice to `answer` only. Watch whether the evolved `answer` instructions pick up your phrasing.

**Next up:** we step off the optimization track: programs whose *graph* is decided at runtime — ReAct agents with tools — plus buying accuracy with inference-time compute (`Refine`, `BestOfN`).